# 🏃 Sports Multi-Object Tracker — Notebook Walkthrough

This notebook walks through the full pipeline step by step:
1. Install dependencies
2. Download a sample video
3. Run detection
4. Run tracking
5. Visualise results
6. Analyse stats

## 1. Install Dependencies

In [ ]:
!pip install ultralytics supervision opencv-python-headless gradio -q

## 2. Download a Sample Public Video

In [ ]:
# Option A: use yt-dlp (install separately)
# !pip install yt-dlp -q
# !yt-dlp -o sample.mp4 --max-filesize 50m 'YOUR_YOUTUBE_URL'

# Option B: use any local video
VIDEO_PATH = 'sample.mp4'   # ← change this to your video path
OUTPUT_PATH = 'output/annotated_video.mp4'

import os
os.makedirs('output/screenshots', exist_ok=True)
print('Paths set.')

## 3. Initialise Detector & Tracker

In [ ]:
import cv2
import supervision as sv
import matplotlib.pyplot as plt
import numpy as np

from detector   import PersonDetector
from tracker    import SportsTracker
from visualizer import FrameVisualizer, HeatmapAccumulator

cap = cv2.VideoCapture(VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {width}x{height} @ {fps:.1f} fps  |  {total} frames')

detector   = PersonDetector('yolov8n.pt', conf_threshold=0.30)
tracker    = SportsTracker(fps=fps, traj_len=60)
visualizer = FrameVisualizer(show_trajectories=True, show_speed=True)
heatmap    = HeatmapAccumulator((height, width))

## 4. Inspect a Single Frame

In [ ]:
# Jump to frame 100 and show raw detections
cap.set(cv2.CAP_PROP_POS_FRAMES, 100)
ret, frame = cap.read()

detections = detector.detect(frame)
print(f'Detections on frame 100: {len(detections)}')
print(detections.xyxy[:5])   # first 5 boxes

# Quick visualise
preview = sv.BoxAnnotator().annotate(frame.copy(), detections)
plt.figure(figsize=(12,6))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.title('Raw Detections — Frame 100')
plt.axis('off')
plt.tight_layout()
plt.savefig('output/screenshots/frame100_detections.png', dpi=150)
plt.show()

## 5. Run Full Pipeline & Save Output Video

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)   # reset to start

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter('output/_raw.mp4', fourcc, fps, (width, height))

counts_over_time = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    detections = detector.detect(frame)
    tracked    = tracker.update(detections)
    heatmap.add(tracked)

    annotated  = visualizer.annotate(frame, tracked, tracker.state, frame_idx)
    writer.write(annotated)

    counts_over_time.append(len(tracked))
    frame_idx += 1

    if frame_idx % 60 == 0:
        print(f'Frame {frame_idx}/{total} — Subjects: {len(tracked)}')

cap.release()
writer.release()

# Re-encode
import os
os.system(f'ffmpeg -y -i output/_raw.mp4 -vcodec libx264 -crf 23 -preset fast -movflags +faststart {OUTPUT_PATH} -loglevel error')
print(f'✅ Done. Unique IDs: {len(tracker.state.unique_ids)}')

## 6. Analyse Results

In [ ]:
# Subject count over time
plt.figure(figsize=(14, 4))
plt.plot(counts_over_time, color='#00d4ff', linewidth=1.5)
plt.fill_between(range(len(counts_over_time)), counts_over_time, alpha=0.2, color='#00d4ff')
plt.title('Subjects Detected Per Frame', fontsize=14)
plt.xlabel('Frame')
plt.ylabel('Count')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/screenshots/count_over_time.png', dpi=150)
plt.show()

# Heatmap
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(heatmap.render(), cv2.COLOR_BGR2RGB))
plt.title('Movement Heatmap')
plt.axis('off')
plt.tight_layout()
plt.savefig('output/screenshots/heatmap.png', dpi=150)
plt.show()

print(f'Total frames   : {frame_idx}')
print(f'Unique IDs     : {len(tracker.state.unique_ids)}')
print(f'Avg subjects/f : {sum(counts_over_time)/len(counts_over_time):.1f}')